# Exercise 2: video → frames → JSON timeline

## Tasks

1. **Frames:** cut a short clip into single pictures.
2. **Describe:** let the API describe one frame – then all of them.
3. **Listen:** turn the speech into text with start and end times.
4. **Merge:** put pictures and speech on one timeline.

## Setup

Select the **Python 3.11** kernel. The API key comes from your `OPENAI_API_KEY` Codespaces secret.

In [ ]:
# Standard Python libraries.
import base64
import json
import mimetypes
import os
from pathlib import Path

# Libraries installed from requirements.txt.
import cv2
import matplotlib.pyplot as plt
import pandas as pd
from openai import OpenAI
from PIL import Image

# Work inside this exercise folder, wherever Jupyter starts.
if not Path("data").exists():  # we are inside an exercise folder
    os.chdir("..")  # go to the main folder
os.chdir("02_Video_Timeline")

# Connect to OpenAI. The key is read from OPENAI_API_KEY; busy requests are retried.
client = OpenAI(max_retries=5, timeout=60)
MODEL = "gpt-6-luna"

# Folder with the course images and videos.
DATA = Path("..") / "data"

## Helper functions for the API calls

- `image_to_input(path)` turns an image file into text (base64), because the API request is sent as text.
- `describe(prompt, path)` sends the prompt and one image, writes a log line to `api_log.jsonl` and returns the answer text.

In [ ]:
# Function 1: turn an image file into an API input.
def image_to_input(path):
    image_type = mimetypes.guess_type(path)[0]  # e.g. "image/jpeg"
    image_text = base64.b64encode(path.read_bytes()).decode()  # bytes -> text
    return {
        "type": "input_image",
        "image_url": f"data:{image_type};base64,{image_text}",
        "detail": "low",  # low detail = fewer tokens = cheaper
    }


# Function 2: send a prompt and one image, get the answer text.
def describe(prompt, path):
    # 1. Put the text and the image into one message.
    content = [{"type": "input_text", "text": prompt}, image_to_input(path)]

    # 2. Send the message.
    response = client.responses.create(
        model=MODEL,
        input=[{"role": "user", "content": content}],
        reasoning={"effort": "none"},
        max_output_tokens=200,
        store=False,
    )

    # 3. Log prompt, model and raw answer.
    entry = {"prompt": prompt, "model": MODEL, "answer": response.output_text}
    with open("api_log.jsonl", "a", encoding="utf-8") as log:
        log.write(json.dumps(entry, ensure_ascii=False) + "\n")

    # 4. Return the answer text.
    return response.output_text

## 1. Cut the clip into one picture per second

The frames are saved in `frames/` – you can open them in the Explorer.

In [ ]:
# The video to analyse.
video_path = DATA / "videos" / "ueli_scene3_talk.mp4"

# Create the frames folder and open the video.
Path("frames").mkdir(exist_ok=True)
video = cv2.VideoCapture(str(video_path))

# Save one frame per second as a JPG file.
paths = []
for t in range(8):  # Flow clips are 8 seconds long
    video.set(cv2.CAP_PROP_POS_MSEC, t * 1000)  # jump to second t
    ok, frame = video.read()
    if not ok:  # the video ended earlier
        break
    path = Path("frames") / f"frame_{t}s.jpg"
    cv2.imwrite(str(path), frame)
    paths.append(path)

# Close the video file.
video.release()
print(len(paths), "frames saved in frames/")

In [ ]:
# Show all frames in one row, with their time.
fig, axes = plt.subplots(1, len(paths), figsize=(2 * len(paths), 2))
for t in range(len(paths)):
    axes[t].imshow(Image.open(paths[t]))
    axes[t].set_title(f"{t} s")
    axes[t].axis("off")
plt.show()

## 2. Let the API describe one frame – then all of them

While the cells run: create your Google account for Flow.

In [ ]:
# The instruction for the model.
PROMPT = "Describe this video frame in one short sentence. Does Ueli look toward the camera?"

# Try it on the first frame.
print(describe(PROMPT, paths[0]))

In [ ]:
# Describe every frame and add the time ourselves.
rows = []
for t in range(len(paths)):
    rows.append({"t": t, "description": describe(PROMPT, paths[t])})

# One row per frame.
frames = pd.DataFrame(rows)
frames

## 3. Turn the speech into text with start and end times

Swiss German often comes back as Standard German.

In [ ]:
# Send the video to Whisper. It reads the sound from the mp4.
with open(video_path, "rb") as audio:
    response = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio,
        response_format="verbose_json",
        timestamp_granularities=["segment"],
    )

# Add one line to the log file.
entry = {"prompt": "transcribe", "model": "whisper-1", "answer": response.text}
with open("api_log.jsonl", "a", encoding="utf-8") as log:
    log.write(json.dumps(entry, ensure_ascii=False) + "\n")

# One row per speech segment: start, end and text.
speech = pd.DataFrame(response.model_dump()["segments"])[["start", "end", "text"]]
speech

## 4. Put pictures and speech on one timeline

A frame belongs to a speech segment when `start ≤ t < end`. The time is the key that joins what is seen and what is said.

In [ ]:
# Copy the text of each speech segment to the frames inside [start, end).
frames["said"] = ""
for i, segment in speech.iterrows():
    inside = frames["t"].between(segment["start"], segment["end"], inclusive="left")
    frames.loc[inside, "said"] = segment["text"]

# Show what is seen and what is said at the same moment.
frames

A short line between two frames is missing in this table. That is why the saved timeline keeps every frame **and** every speech segment as its own row.

In [ ]:
# One row per frame: it covers one second.
pictures = pd.DataFrame({
    "start": frames["t"],
    "end": frames["t"] + 1,
    "kind": "picture",
    "text": frames["description"],
})

# One row per speech segment, with its own start and end.
spoken = pd.DataFrame({
    "start": speech["start"],
    "end": speech["end"],
    "kind": "speech",
    "text": speech["text"],
})

# Put both on one timeline, sorted by time, and save it as JSON lines.
timeline = pd.concat([pictures, spoken]).sort_values("start")
timeline.to_json("timeline.jsonl", orient="records", lines=True, force_ascii=False)
timeline

**Check yourself:** What do you lose with one frame every 2 seconds instead of every second? (Hint: «Lueg, Bruno»)

### Jupyter notebook --footer info--

In [ ]:
# Print system and version info for this run.
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')